In [5]:
from ultralytics import YOLO


# Load the exported ONNX model
onnx_model = YOLO(r"ultralytics\best.onnx",task="detect")

# Run inference
results = onnx_model(r"ultralytics\dataset\images\val\pitted_surface_77.jpg")

Loading ultralytics\best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.23.2 with CUDAExecutionProvider

image 1/1 d:\A-project\YOLO-for-NEU-DET-dataset\ultralytics\dataset\images\val\pitted_surface_77.jpg: 224x224 2 pitted_surfaces, 57.5ms
Speed: 1.2ms preprocess, 57.5ms inference, 1.2ms postprocess per image at shape (1, 3, 224, 224)


In [10]:
results

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'crazing', 1: 'inclusion', 2: 'patches', 3: 'pitted_surface', 4: 'rolled-in_scale', 5: 'scratches'}
 obb: None
 orig_img: array([[[242, 242, 242],
         [241, 241, 241],
         [231, 231, 231],
         ...,
         [197, 197, 197],
         [202, 202, 202],
         [207, 207, 207]],
 
        [[244, 244, 244],
         [239, 239, 239],
         [233, 233, 233],
         ...,
         [205, 205, 205],
         [209, 209, 209],
         [205, 205, 205]],
 
        [[240, 240, 240],
         [235, 235, 235],
         [235, 235, 235],
         ...,
         [199, 199, 199],
         [203, 203, 203],
         [198, 198, 198]],
 
        ...,
 
        [[231, 231, 231],
         [240, 240, 240],
         [228, 228, 228],
         ...,
         [189, 189, 189],
         [190, 190, 190],
         [189, 189, 189]],
 
        [[230, 230, 

In [14]:
import onnxruntime as ort
import numpy as np
from PIL import Image

# 加载 ONNX 模型
session = ort.InferenceSession(r"ultralytics\best.onnx", providers=['CPUExecutionProvider'])

# 预处理图片 (以 YOLO 默认的 640x640 为例)
img = Image.open(r"ultralytics\dataset\images\val\pitted_surface_77.jpg").resize((224, 224))
img_array = np.array(img).transpose(2, 0, 1)[np.newaxis, ...].astype(np.float32) / 255.0

# 推理
outputs = session.run(None, {"images": img_array})

In [22]:
outputs[0][0] 

array([[   -0.82774,    0.005127,      132.65,      223.95,     0.88733,           3],
       [     137.83,       1.465,      224.85,       142.3,     0.67083,           3],
       [     149.74,      171.79,      225.11,      224.05,     0.15114,           3],
       ...,
       [     101.85,      217.11,       168.1,      224.04,   6.333e-05,           2],
       [     174.01,   -0.081032,      223.18,      25.279,  6.3062e-05,           4],
       [     137.14,      16.548,      225.87,      186.56,  6.2943e-05,           4]], shape=(300, 6), dtype=float32)

In [21]:
predictions = outputs[0][0]  # 输出形状: (300, 6)
confidence_threshold = 0.5
valid_predictions = predictions[predictions[:, 4] > confidence_threshold]

# 现在，valid_predictions 中就是最终检测结果，可以直接使用
print(valid_predictions)

[[   -0.82774    0.005127      132.65      223.95     0.88733           3]
 [     137.83       1.465      224.85       142.3     0.67083           3]]
